# Trading backtest and a reward-to-risk policy

**Track:** Portfolio. **Level:** Practitioner. **CPU:** yes.

This lab uses a synthetic geometric Brownian path. It is a software exercise in backtesting and in optimizing a risk-adjusted reward. It is not a trading strategy, not investment advice, and not evidence that the learned weights will make money on real prices.

## Goals

- Check a backtest against a hand-computed profit on a three-point price path.
- Include a transaction cost proportional to turnover.
- Fit a small linear policy by direct search on a Sharpe reward, using only past features.
- Report the test window separately from the training window.

## Sources and license

Original notebook. The reward is the Sharpe ratio, following the use of risk-adjusted return as a training signal in Moody and Saffell, *Learning to Trade via Direct Reinforcement* (2001). Position sizing is a squashed linear function of trailing momentum and volatility. No market data is downloaded.

## Concepts

A backtest applies yesterday's position to today's return. Using today's return to choose today's position is leakage. Turnover is the absolute change in position. Cost is turnover times a rate in basis points. Sharpe here is `sqrt(252) * mean(pnl) / std(pnl)` on the window you pass in. Optimizing it on the training slice and then looking at the test slice is the whole point of the harness. A higher training Sharpe is not a result about markets. It is a check that the optimizer moved.


In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", 'numpy', 'pandas', 'matplotlib'])
print("packages ready")


In [ ]:
import os
import sys
from pathlib import Path

def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for cand in [start, *start.parents]:
        if (cand / "labs" / "_shared" / "llm.py").exists():
            return cand
    if "google.colab" in sys.modules:
        root = Path("/content/ShaiLearnsAI")
        marker = root / "labs" / "_shared" / "llm.py"
        if not marker.exists():
            os.system(
                "git clone --depth 1 https://github.com/shai200/ShaiLearnsAI.git /content/ShaiLearnsAI"
            )
        return root
    raise RuntimeError(
        "Clone https://github.com/shai200/ShaiLearnsAI and run this notebook from that directory."
    )

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
print("repo", ROOT)

%matplotlib inline
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from labs._shared.seed import seed_everything

seed_everything(0)
rng = np.random.default_rng(0)


## Guided exercise: the harness

`position[t]` is decided from prices up to `t` and earns `price[t+1] / price[t] - 1`. The identity test is a price path 100, 110, 121 with a constant long position and no cost. Both period returns are 0.10.


In [ ]:
def backtest(price: np.ndarray, position: np.ndarray, cost_bps: float = 0.0):
    ret = np.diff(price) / price[:-1]
    pos = np.asarray(position, dtype=float)[: len(ret)]
    turnover = np.abs(np.diff(pos, prepend=0.0))
    pnl = pos * ret - turnover * (cost_bps / 1e4)
    equity = np.cumprod(1.0 + pnl)
    return pnl, equity

def sharpe(pnl: np.ndarray) -> float:
    if len(pnl) < 2 or float(np.std(pnl)) < 1e-12:
        return 0.0
    return float(math.sqrt(252) * np.mean(pnl) / np.std(pnl))

def max_drawdown(equity: np.ndarray) -> float:
    peak = np.maximum.accumulate(equity)
    return float((equity / peak - 1.0).min())

identity_pnl, _ = backtest(np.array([100.0, 110.0, 121.0]), np.array([1.0, 1.0]), cost_bps=0.0)
print("identity pnl", identity_pnl.round(6))
assert np.allclose(identity_pnl, [0.1, 0.1])

costly, _ = backtest(np.array([100.0, 110.0, 121.0]), np.array([1.0, 1.0]), cost_bps=50.0)
assert costly.sum() < identity_pnl.sum()


## Guided exercise: features and the policy

Momentum is the mean of the last 10 returns. Volatility is their standard deviation. Both are computed from returns that end at `t`, so they do not include the return the position is about to earn. The policy is `tanh(w · [momentum, volatility, 1])`. Weights are fit by coordinate search on training Sharpe. That is direct policy search on a scalar reward, small enough to read.


In [ ]:
def synthetic_price(n: int = 500, mu: float = 0.0003, sigma: float = 0.012) -> np.ndarray:
    shocks = rng.normal(mu, sigma, size=n)
    return 100.0 * np.exp(np.cumsum(shocks))

def feature_matrix(price: np.ndarray) -> np.ndarray:
    ret = np.zeros(len(price))
    ret[1:] = np.diff(price) / price[:-1]
    mom = np.zeros(len(price))
    vol = np.zeros(len(price))
    for t in range(11, len(price)):
        window = ret[t - 10 : t]
        mom[t] = window.mean()
        vol[t] = window.std() + 1e-6
    return np.column_stack([mom, vol, np.ones(len(price))])

def positions_of(features: np.ndarray, weights: np.ndarray) -> np.ndarray:
    return np.tanh(features @ weights)

def window_sharpe(price, features, weights, sl, cost_bps):
    pos = positions_of(features, weights)
    pnl, _ = backtest(price, pos, cost_bps=cost_bps)
    return sharpe(pnl[sl])

def fit_policy(price, features, sl, cost_bps, objective="sharpe"):
    weights = np.zeros(3)
    step = 0.4

    def score(w):
        pos = positions_of(features, w)
        pnl, equity = backtest(price, pos, cost_bps=cost_bps)
        segment = pnl[sl]
        if objective == "sharpe":
            return sharpe(segment)
        return float(np.mean(segment))

    best = score(weights)
    for _ in range(25):
        improved = False
        for i in range(len(weights)):
            for direction in (step, -step):
                trial = weights.copy()
                trial[i] += direction
                value = score(trial)
                if value > best + 1e-8:
                    weights, best, improved = trial, value, True
        if not improved:
            step *= 0.5
            if step < 0.05:
                break
    return weights, best

price = synthetic_price()
features = feature_matrix(price)
train = slice(20, 320)
test = slice(320, 499)
weights, train_score = fit_policy(price, features, train, cost_bps=5.0, objective="sharpe")
print("weights", weights.round(3), "train sharpe", round(train_score, 3))

def report(name, pos):
    pnl, equity = backtest(price, pos, cost_bps=5.0)
    return {
        "policy": name,
        "test sharpe": sharpe(pnl[test]),
        "test mean pnl": float(pnl[test].mean()),
        "test max drawdown": max_drawdown(equity[test]),
        "mean abs position": float(np.abs(pos[test]).mean()),
    }

learned = positions_of(features, weights)
momentum = np.tanh(40.0 * features[:, 0])
flat = np.ones(len(price))
table = pd.DataFrame([
    report("reward-to-risk", learned),
    report("momentum sign", momentum),
    report("always long", flat),
])
print(table.round(3).to_string(index=False))
assert np.isfinite(table["test sharpe"]).all()
assert train_score >= sharpe(backtest(price, positions_of(features, np.zeros(3)), 5.0)[0][train]) - 1e-8

pnl, equity = backtest(price, learned, cost_bps=5.0)
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(equity, label="reward-to-risk")
long_equity = backtest(price, flat, cost_bps=5.0)[1]
ax.plot(long_equity, label="always long")
ax.axvline(320, color="gray", ls="--", lw=1)
ax.legend()
ax.set_title("Synthetic equity, cost 5 bps. Line is the test split.")
plt.tight_layout()
plt.show()
print("LAB OK")


## Your turn

Recompute the identity path with a short position of -1. The profit should be -0.10, then -0.10.

## Solution


In [ ]:
short_pnl, _ = backtest(np.array([100.0, 110.0, 121.0]), np.array([-1.0, -1.0]))
print(short_pnl.round(6))
assert np.allclose(short_pnl, [-0.1, -0.1])


## Beyond the course

Fit the same policy on mean profit instead of Sharpe, on the same training slice and the same cost. Compare mean absolute position on the test slice. The risk-adjusted objective is allowed to take smaller positions. The table is the result. It is still a result about this synthetic path.


In [ ]:
return_weights, _ = fit_policy(price, features, train, cost_bps=5.0, objective="mean")
return_pos = positions_of(features, return_weights)
print(pd.DataFrame([
    report("sharpe objective", learned),
    report("mean-pnl objective", return_pos),
]).round(3).to_string(index=False))
assert np.isfinite(return_pos).all()


## Research questions

1. The test window is one path. What would you need before treating a Sharpe difference as stable?
2. The feature at time `t` uses returns up to `t-1` only inside `feature_matrix`. Where would a one-line off-by-one put today's return into the feature?
3. Cost is linear in turnover. Which policy in the table is punished most if the cost goes from 5 bps to 50 bps, and how would you measure it?

## Further reading

- Moody and Saffell, *Learning to Trade via Direct Reinforcement*, IEEE Transactions on Neural Networks, 2001.
- Sharpe, *The Sharpe Ratio*, Journal of Portfolio Management, 1994.
- Sutton and Barto, *Reinforcement Learning: An Introduction*, for policy search. The book is a reference, not copied.
